# 03 · Strategia ottimale
Stima degrado (con correzione carburante) e calcola la strategia migliore.

In [ ]:
import sys; sys.path.insert(0, '../src')
from f1va import data as f1data, features, models, strategy, synthetic
import pandas as pd, numpy as np

# Offline di default (nessuna rete). Metti False per usare dati reali FastF1.
USE_SYNTHETIC = True

def load_laps(year=2024, gp='Monza', laps=53):
    if USE_SYNTHETIC:
        return synthetic.generate_race(n_drivers=20, laps=laps, seed=0)
    ses = f1data.load_session(year, gp, 'R')
    return f1data.quicklaps(f1data.laps_dataframe(ses))

In [ ]:
laps = load_laps(laps=53)
deg = features.fuel_corrected_degradation(laps)
tyre = strategy.fit_tyre_models(deg)
deg

## Strategia ottimale

In [ ]:
best = strategy.optimize_strategy(53, list(tyre.keys()), tyre, max_stops=2)
print('Piano:', ' -> '.join(f'{c}({n})' for c,n in best.stints))
print('Tempo gara stimato:', round(best.total_time_s/60,2), 'min')

## Undercut

In [ ]:
keys=list(tyre.keys())
print('Undercut gain (s):', strategy.undercut_gain(tyre, keys[0], rival_deg_compound=keys[-1]))